In [1]:
import matminer
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df_sc_ef = pd.read_csv("df_sc_ef_ElementFraction_ftd.csv").dropna()
df_sc_ep = pd.read_csv("df_sc_ep_ElementProperty_Magpie_ftd.csv").dropna()

In [3]:
df_sc_ef = df_sc_ef.iloc[:, 1:]
df_sc_ep = df_sc_ep.iloc[:, 1:]

In [4]:
print(df_sc_ef.shape)
print(df_sc_ep.shape)

(16375, 105)
(16375, 134)


In [5]:
df_sc_ef

,Critical Temp,_Composition,H,He,Li,Be,B,C,N,O,...,Pu,Am,Cm,Bk,Cf,Es,Fm,Md,No,Lr
0,31.20,Ba0.4 K0.6 Fe2 As2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.541925,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6.90,Mo0.39 Ru0.61,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.10,Tm4 Os6 Sn19,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4.85,Nd1 Bi0.99 Pb0.01 S2 F0.3 O0.7,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.140000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16409,0.00,Al4 C3,0.0,0.0,0.0,0.0,0.0,0.428571,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16410,8.87,Nb0.96 Ta0.04,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16411,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16412,34.80,Yb0.5 Pr0.5 Ba2 Cu3 O6.9,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.534884,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
merged_df = pd.merge(df_sc_ef, df_sc_ep, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="outer")
merged_df

,Critical Temp,_Composition,H,He,Li,Be,B,C,N,O,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,31.20,Ba0.4 K0.6 Fe2 As2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.541925,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
2,6.90,Mo0.39 Ru0.61,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,194.0,229.0,35.0,207.650000,16.653000,194.0
3,1.10,Tm4 Os6 Sn19,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,141.0,194.0,53.0,159.275862,23.947681,141.0
4,4.85,Nd1 Bi0.99 Pb0.01 S2 F0.3 O0.7,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.140000,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,72.206000,49.328776,70.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16386,0.00,Al4 C3,0.0,0.0,0.0,0.0,0.0,0.428571,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,194.0,225.0,31.0,211.714286,15.183673,225.0
16387,8.87,Nb0.96 Ta0.04,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,229.0,229.0,0.0,229.000000,0.000000,229.0
16388,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.500000,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0
16389,34.80,Yb0.5 Pr0.5 Ba2 Cu3 O6.9,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.534884,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,110.488372,105.359654,12.0


In [7]:
merged_df["having_tc"] = (merged_df["Critical Temp"] >= 10).astype(int)
merged_df

,Critical Temp,_Composition,H,He,Li,Be,B,C,N,O,...,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,having_tc
0,31.20,Ba0.4 K0.6 Fe2 As2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0,1
1,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.541925,...,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0,1
2,6.90,Mo0.39 Ru0.61,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.0,194.0,229.0,35.0,207.650000,16.653000,194.0,0
3,1.10,Tm4 Os6 Sn19,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.0,141.0,194.0,53.0,159.275862,23.947681,141.0,0
4,4.85,Nd1 Bi0.99 Pb0.01 S2 F0.3 O0.7,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.140000,...,0.000000,0.000000,0.0,12.0,225.0,213.0,72.206000,49.328776,70.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16386,0.00,Al4 C3,0.0,0.0,0.0,0.0,0.0,0.428571,0.0,0.000000,...,0.000000,0.000000,0.0,194.0,225.0,31.0,211.714286,15.183673,225.0,0
16387,8.87,Nb0.96 Ta0.04,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.000000,0.000000,0.0,229.0,229.0,0.0,229.000000,0.000000,229.0,0
16388,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.500000,...,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0,1
16389,34.80,Yb0.5 Pr0.5 Ba2 Cu3 O6.9,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.534884,...,0.000000,0.000000,0.0,12.0,229.0,217.0,110.488372,105.359654,12.0,1


In [8]:
X = merged_df.iloc[:, 2:-1]
y = merged_df['having_tc']

X

,H,He,Li,Be,B,C,N,O,F,Ne,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00,0.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.541925,0.00,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00,0.0,...,0.000000,0.000000,0.000000,0.0,194.0,229.0,35.0,207.650000,16.653000,194.0
3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00,0.0,...,0.000000,0.000000,0.000000,0.0,141.0,194.0,53.0,159.275862,23.947681,141.0
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.140000,0.06,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,72.206000,49.328776,70.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16386,0.0,0.0,0.0,0.0,0.0,0.428571,0.0,0.000000,0.00,0.0,...,0.000000,0.000000,0.000000,0.0,194.0,225.0,31.0,211.714286,15.183673,225.0
16387,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00,0.0,...,0.000000,0.000000,0.000000,0.0,229.0,229.0,0.0,229.000000,0.000000,229.0
16388,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.500000,0.00,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0
16389,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.534884,0.00,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,110.488372,105.359654,12.0


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

['H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne', 'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'I', 'Xe', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'At', 'Rn', 'Fr', 'Ra', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm', 'Md', 'No', 'Lr', 'MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number', 'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number', 'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber', 'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber', 'MagpieData avg_dev MendeleevNumber', 'MagpieData mode MendeleevNumber', 'MagpieDa

235

In [10]:
# param_grid = {
#     "n_estimators": [100, 200, 500],  # Number of trees in the forest
#     "max_depth": [5, 10, 15],        # Maximum depth of individual trees
#     "min_samples_split": [2, 5, 10],  # Minimum samples required to split a node
# }

param_grid = {
    "n_estimators": [100, 200, 500, 1000],  # Increase the number of trees
    "max_depth": [10, 15, 20],               # Expand the maximum depth
    "min_samples_split": [2, 5, 10, 20],     # Add more values for minimum samples split
    "min_samples_leaf": [1, 2, 4]            # Introduce minimum samples required for a leaf node
}


In [11]:
# Create the Random Forest classifier
model = RandomForestClassifier()

# Create the GridSearchCV object
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1, verbose=2)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'max_depth': [10, 15, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10, 20],
                         'n_estimators': [100, 200, 500, 1000]},
             scoring='accuracy', verbose=2)

In [12]:
# Get the best model from the grid search
best_model = grid_search.best_estimator_

# Predict on the testing set
y_pred = best_model.predict(X_test)

In [13]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)

# Precision
precision = precision_score(y_test, y_pred)
print("Precision:", precision)

# Recall
recall = recall_score(y_test, y_pred)
print("Recall:", recall)

# F1 Score
f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred)
print("ROC-AUC Score:", roc_auc)

Accuracy: 0.9153245485602733
Confusion Matrix:
 [[2327  221]
 [ 126 1424]]
Precision: 0.8656534954407294
Recall: 0.9187096774193548
F1 Score: 0.8913928012519562
ROC-AUC Score: 0.9159874917709019


In [14]:
filename = 'sc_efep_rf_cl.pkl'
with open(filename, 'wb') as f:
    pickle.dump(best_model, f)

In [15]:
# Accuracy: 0.9111761835041484
# Confusion Matrix:
#  [[2306  242]
#  [ 122 1428]]
# Precision: 0.8550898203592814
# Recall: 0.9212903225806451
# F1 Score: 0.8869565217391303
# ROC-AUC Score: 0.9131569352306679

In [ ]:
# 720 fits

# Accuracy: 0.9153245485602733
# Confusion Matrix:
#  [[2327  221]
#  [ 126 1424]]
# Precision: 0.8656534954407294
# Recall: 0.9187096774193548
# F1 Score: 0.8913928012519562
# ROC-AUC Score: 0.9159874917709019